# CodeAlpha Task 1 — Credit Scoring Model

Predict an individual's creditworthiness using past financial data. This notebook follows the CodeAlpha requirements: financial-history feature engineering, classification models, and Precision, Recall, F1-Score and ROC-AUC evaluation.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))
import pandas as pd
from preprocess import add_engineered_features
from config import DATA_PATH
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()
df.describe(include='all').T

In [ ]:
df = add_engineered_features(df)
df[['income','debt','loan_amount','debt_to_income','loan_to_income','savings_to_income','creditworthy']].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from preprocess import build_preprocessor

X = df.drop(columns=['creditworthy'])
y = df['creditworthy'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=8, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=10, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1),
}

rows=[]
for name, model in models.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob),
    })
pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False)

## Conclusion

The model with the highest ROC-AUC is selected as the best credit scoring model. The project demonstrates financial-history feature engineering and classification-based creditworthiness prediction.